# Validação reproduzível da NexoPlan

Experiência de verificação de **dados sintéticos, Média Móvel, ETS(A,A,N), MAE, MAPE e cobertura**.

**Objetivo:** comparar cálculos de referência em Python com as classes Java reais, sem iniciar o servidor nem alterar a base de dados.

- Fontes: `data/empresa/`, motores em `backend/src/main/java/` e fixture `backend/src/test/resources/ets/reference-statsmodels.json`.
- Critérios: dados íntegros; média móvel e métricas iguais à precisão publicada; ETS consistente com referências; casos temporais explicáveis.
- Os resultados são relativos aos ficheiros da experiência. Não certificam a base de dados em execução nem substituem testes REST, autorização ou desempenho.

## 1. Preparação

Requer Python 3.10+ e Java 17+. Antes da comparação Java, compila o backend na raiz do projeto:

```bash
cd backend
./mvnw -q -DskipTests compile
cd ..
```

Todas as células usam apenas a biblioteca padrão Python. Executa-as pela ordem apresentada. A semente e os cortes são fixos; não usamos o relógio atual.

In [1]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP, localcontext
from datetime import datetime, timezone
from collections import Counter, defaultdict, deque
import csv, hashlib, json, math, random, re, shutil, subprocess, sys, tempfile

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "backend/pom.xml").is_file() and (p / "data/empresa/manifesto.json").is_file()), None)
if REPO is None:
    raise RuntimeError("Abre o notebook dentro da pasta sales-operations-planner.")
DATA = REPO / "data/empresa"
CLASSES = REPO / "backend/target/classes"
SCALE = Decimal("0.0001")
SEED = 20261112
checks = []

def check(condition, description):
    if not condition:
        raise AssertionError(description)
    checks.append(description)

def dec(value):
    return None if value is None else Decimal(str(value))

def q4(value):
    return None if value is None else format(dec(value).quantize(SCALE, rounding=ROUND_HALF_UP), ".4f")

def utc(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00"))

def month_start(value):
    return datetime.strptime(value, "%Y-%m").replace(tzinfo=timezone.utc)

def plus_month(value, n=1):
    year, month = map(int, value.split("-"))
    absolute = year * 12 + month - 1 + n
    return f"{absolute // 12:04d}-{absolute % 12 + 1:02d}"

def read_csv(name):
    with (DATA / name).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream, delimiter=";"))

print("Ambiente: Python", sys.version.split()[0], "| Semente:", SEED)

Ambiente: Python 3.14.10 | Semente: 20261112


## 2. Integridade dos ficheiros da empresa fictícia

Verificamos hashes do manifesto, contagens, chaves únicas, referências ao catálogo, unidades e restrições operacionais. As duas faltas documentadas são intencionais: um realizado ausente e um stock de fecho vazio.

In [2]:
manifest = json.loads((DATA / "manifesto.json").read_text(encoding="utf-8"))
check(manifest["sintetico"] is True, "O manifesto identifica dados sintéticos")
for name, expected in manifest["sha256"].items():
    check(hashlib.sha256((DATA / name).read_bytes()).hexdigest() == expected, f"SHA-256: {name}")
files = {name: read_csv(name) for name in manifest["contagens"]}
for name, count in manifest["contagens"].items():
    check(len(files[name]) == count, f"Contagem: {name}")
products = {row["codigo"]: row for row in files["produtos.csv"]}
check(len(products) == manifest["produtos"], "Códigos de produto únicos")
for name, rows in files.items():
    if name == "produtos.csv":
        continue
    keys = [(r["produto"], r["periodo"], r.get("tipo", "")) for r in rows]
    check(len(set(keys)) == len(keys), f"Chaves produto/mês/tipo únicas: {name}")
    check(all(r["produto"] in products and r["unidade"] == products[r["produto"]]["unidade"] for r in rows),
          f"Referências e unidades: {name}")
    check(all(re.fullmatch(r"[1-9][0-9]{3}-(0[1-9]|1[0-2])", r["periodo"]) for r in rows),
          f"Meses válidos: {name}")
    numeric_fields = [k for k in rows[0] if k not in {"produto", "periodo", "tipo", "unidade", "disponivel_em"}]
    check(all(Decimal(r[k]) >= 0 for r in rows for k in numeric_fields if r[k] != ""),
          f"Quantidades observadas não negativas: {name}")
    check(all(utc(r["disponivel_em"]).tzinfo is not None for r in rows), f"Disponibilidade com fuso: {name}")
actual = {(r["produto"], r["periodo"]): Decimal(r["quantidade"]) for r in files["realizados.csv"]}
observed = {(r["produto"], r["periodo"]): Decimal(r["quantidade"])
            for r in files["historico_atual.csv"] if r["tipo"] == "VENDAS"}
check(all(observed[key] == value for key, value in actual.items()), "Vendas e realizados coincidem nos meses comuns")
ops = files["operacionais.csv"]
check(all(Decimal(r["unidades_satisfeitas_no_prazo"]) <= Decimal(r["unidades_solicitadas"]) for r in ops),
      "Serviço: satisfeitas não excedem solicitadas")
check(all(int(r["dias_com_procura_sem_stock"]) <= int(r["dias_observados_com_procura"]) for r in ops),
      "Rutura: dias sem stock não excedem dias observados")
expected_keys = {(p, f"2026-{m:02d}") for p in products for m in range(7, 11)}
check(expected_keys - set(actual) == {("NX-005", "2026-10")}, "Única falta de realizado é a documentada")
check([(r["produto"], r["periodo"]) for r in ops if not r["stock_fecho"]] == [("NX-006", "2026-10")],
      "Único stock de fecho vazio é o documentado")
for name, rows in files.items():
    print(f"{name:23} {len(rows):4} linhas")
print("Unidades:", dict(Counter(p["unidade"] for p in products.values())))

produtos.csv              24 linhas
historico_treino.csv     672 linhas
historico_atual.csv      672 linhas
realizados.csv            95 linhas
operacionais.csv          96 linhas
Unidades: {'un': 19, 'kg': 3, 'L': 2}


## 3. Média Móvel: referência independente

Cada passo usa a janela anterior e inclui a previsão recém-calculada. Arredondamos cada passo a quatro casas, com `HALF_UP`, como o contrato publicado. Os zeros são observações válidas.

In [3]:
def moving_average(values, horizon):
    window = deque(map(dec, values))
    if not window or horizon < 1:
        raise ValueError("Histórico e horizonte inválidos")
    result = []
    with localcontext() as ctx:
        ctx.prec = 50
        for _ in range(horizon):
            value = (sum(window) / len(window)).quantize(SCALE, rounding=ROUND_HALF_UP)
            result.append(value)
            window.popleft()
            window.append(value)
    return result

ma_cases = [
    ("recursiva", [10, 20, 30], 3, ["20.0000", "23.3333", "24.4444"]),
    ("zeros", [10, 0, 20], 3, ["10.0000", "10.0000", "13.3333"]),
    ("arredondamento", [0, 0, 1], 3, ["0.3333", "0.4444", "0.5926"]),
    ("máximo decimal", ["99999999999999.9999"], 12, ["99999999999999.9999"] * 12),
]
for name, history, horizon, expected in ma_cases:
    check(list(map(q4, moving_average(history, horizon))) == expected, f"Referência MM: {name}")
print("Referências conhecidas: 4 casos, incluindo zeros e limite DECIMAL(18,4).")

Referências conhecidas: 4 casos, incluindo zeros e limite DECIMAL(18,4).


## 4. MAE, MAPE e cobertura

- **MAE:** média dos erros absolutos nos pares comuns entre os dois modelos e o realizado; inclui realizados zero.
- **MAPE:** média dos erros percentuais absolutos; exclui apenas os pares cujo realizado é zero.
- **Cobertura:** pares comuns divididos por todas as células candidatas.

Sem pares, MAE/MAPE são indefinidos (`None`), não zero. Valores de MAE com unidades diferentes não devem ser somados.

In [4]:
def common_metrics(pairs):
    candidates = len(pairs)
    valid = [(dec(a), dec(b), dec(y)) for a, b, y in pairs if a is not None and b is not None and y is not None]
    percentage = [(a, b, y) for a, b, y in valid if y != 0]
    with localcontext() as ctx:
        ctx.prec = 50
        mean = lambda values: sum(values) / len(values) if values else None
        mape = lambda side: mean([abs(row[side] - row[2]) / abs(row[2]) * 100 for row in percentage])
        return {
            "candidatos": candidates, "pares": len(valid), "paresMape": len(percentage),
            "zeros": len(valid) - len(percentage), "emFalta": sum(y is None for _, _, y in pairs),
            "semMM": sum(a is None for a, _, _ in pairs), "semETS": sum(b is None for _, b, _ in pairs),
            "cobertura": q4(Decimal(len(valid)) / candidates * 100) if candidates else None,
            "maeMM": q4(mean([abs(a-y) for a, _, y in valid])),
            "maeETS": q4(mean([abs(b-y) for _, b, y in valid])),
            "mapeMM": q4(mape(0)), "mapeETS": q4(mape(1)),
        }

metric_cases = {
    "zeros_e_faltas": [(90, 110, 100), (10, 0, 0), (30, 20, None)],
    "sem_pares_comuns": [(90, None, 100), (None, 100, 100)],
    "apenas_zero": [(5, 0, 0)],
    "media_percentuais": [(20, 10, 10), (110, 100, 100)],
    "sem_candidatos": [],
}
m = common_metrics(metric_cases["zeros_e_faltas"])
check((m["maeMM"], m["maeETS"], m["mapeMM"], m["mapeETS"], m["cobertura"]) ==
      ("10.0000", "5.0000", "10.0000", "10.0000", "66.6667"), "MAE/MAPE/66,6667% com zeros e faltas")
check(common_metrics(metric_cases["sem_pares_comuns"])["maeMM"] is None, "Sem pares: MAE indefinido")
check(common_metrics(metric_cases["apenas_zero"])["mapeMM"] is None, "Realizado zero: MAPE indefinido")
check(common_metrics(metric_cases["apenas_zero"])["maeMM"] == "5.0000", "Realizado zero continua no MAE")
check(common_metrics(metric_cases["media_percentuais"])["mapeMM"] == "55.0000", "MAPE é média dos percentuais")
check(common_metrics([])["cobertura"] is None, "Sem candidatos: cobertura indefinida")
print("Caso com zeros/faltas:", {k: m[k] for k in ["pares", "paresMape", "zeros", "maeMM", "maeETS", "cobertura"]})

Caso com zeros/faltas: {'pares': 2, 'paresMape': 1, 'zeros': 1, 'maeMM': '10.0000', 'maeETS': '5.0000', 'cobertura': '66.6667'}


## 5. ETS(A,A,N) contra a referência guardada

A variante tem erro e tendência aditivos, sem sazonalidade. O ajuste pesquisa 81 combinações: `alpha = 0,1…0,9` e `beta = alpha × beta*`. O menor SSE de treino decide o ajuste; o resultado não garante superioridade fora do treino.

A fixture existente declara **Statsmodels 0.15.0**. Usamos os valores nela guardados; este notebook não executa nem reinstala Statsmodels.

In [5]:
def ets_reference(history, horizon=3, fixed=None):
    values = list(map(float, history))
    if not 6 <= len(values) <= 120 or not 1 <= horizon <= 12 or min(values) < 0:
        raise ValueError("ETS exige 6–120 observações não negativas e horizonte 1–12")
    def smooth(alpha, beta):
        level, trend, sse = values[0], values[1] - values[0], 0.0
        for value in values[1:]:
            fitted = level + trend
            error = value - fitted
            sse += error * error
            level = fitted + alpha * error
            trend += beta * error
        return level, trend, sse
    if fixed:
        alpha, beta = fixed
    else:
        grid = [(a / 10.0, a * b / 100.0) for a in range(1, 10) for b in range(1, 10)]
        alpha, beta = min(grid, key=lambda ab: smooth(*ab)[2])
    level, trend, sse = smooth(alpha, beta)
    predictions = [q4(max(0.0, level + h * trend)) for h in range(1, horizon+1)]
    return {"predictions": predictions, "alpha": alpha, "beta": beta, "sse": sse}

fixture_path = REPO / "backend/src/test/resources/ets/reference-statsmodels.json"
fixture = json.loads(fixture_path.read_text(encoding="utf-8"))
for case in fixture["cases"]:
    fixed = (case["alpha"], case["beta"]) if case["fixed"] else None
    result = ets_reference(case["history"], fixed=fixed)
    check(result["predictions"] == case["forecast"], f"ETS/fixture: {case['name']}")
    check(math.isclose(result["sse"], case["sse"], rel_tol=1e-10, abs_tol=1e-8), f"SSE/fixture: {case['name']}")
check(ets_reference([100, 80, 60, 40, 20, 0])["predictions"] == ["0.0000"] * 3, "ETS trunca negativos a zero")
print(f"ETS: {len(fixture['cases'])} casos da fixture e truncagem em zero confirmados.")

ETS: 8 casos da fixture e truncagem em zero confirmados.


## 6. Cortes temporais e indisponibilidade explicada

São testes de referência das regras de `MonitoringService.create` e do filtro de treino de `BacktestService.create`; a integração SQL/REST é verificada na suite Java.

Uma publicação tem de ocorrer **antes do início do mês**; o mês tem de estar fechado no corte; o realizado tem de existir e estar disponível nesse instante. Alterar valores futuros não deve alterar o treino selecionado.

In [6]:
def exclusion_reason(month, published, cutoff, available):
    if published >= month_start(month):
        return "PLANO_PUBLICADO_TARDIAMENTE"
    if month_start(plus_month(month)) > cutoff:
        return "PERIODO_NAO_FECHADO"
    if available is None:
        return "SEM_REALIZADO"
    if available > cutoff:
        return "REALIZADO_APOS_CORTE"
    return None

publication = utc("2026-06-30T18:00:00Z")
cutoff = utc("2026-08-01T00:00:00Z")
temporal_cases = [
    ("mês fechado", "2026-07", publication, cutoff, cutoff, None),
    ("publicação no início exato", "2026-07", month_start("2026-07"), cutoff, cutoff, "PLANO_PUBLICADO_TARDIAMENTE"),
    ("mês ainda aberto", "2026-08", publication, cutoff, cutoff, "PERIODO_NAO_FECHADO"),
    ("realizado ausente", "2026-07", publication, cutoff, None, "SEM_REALIZADO"),
    ("realizado posterior ao corte", "2026-07", publication, cutoff, utc("2026-08-02T00:00:00Z"), "REALIZADO_APOS_CORTE"),
]
for label, month, published, cut, available, expected in temporal_cases:
    check(exclusion_reason(month, published, cut, available) == expected, f"Regra temporal: {label}")

def select_training(rows, last_month, cut):
    return sorted((r for r in rows if r["periodo"] <= last_month and utc(r["disponivel_em"]) <= cut),
                  key=lambda r: r["periodo"])

sample = [{"periodo": "2026-05", "quantidade": "10", "disponivel_em": "2026-05-31T17:00:00Z"},
          {"periodo": "2026-06", "quantidade": "20", "disponivel_em": "2026-08-01T00:00:00Z"},
          {"periodo": "2026-07", "quantidade": "30", "disponivel_em": "2026-08-01T00:00:00Z"}]
selected = select_training(sample, "2026-06", month_start("2026-07"))
check([r["periodo"] for r in selected] == ["2026-05"], "Treino exclui futuro e observações disponíveis tarde")
mutated = [dict(row, quantidade="999999") if row["periodo"] > "2026-05" else row for row in sample]
check(select_training(mutated, "2026-06", month_start("2026-07")) == selected, "Alteração de futuro não muda o treino")
print("5 motivos/limites temporais e 2 verificações de isolamento temporal confirmados.")

5 motivos/limites temporais e 2 verificações de isolamento temporal confirmados.


## 7. Ensaio com 24 produtos: julho–setembro de 2026

Treinamos até junho: Média Móvel nos últimos três meses e ETS nos últimos 24. O horizonte é de três meses. Os realizados são exclusivamente os CSV sintéticos.

Este é **um corte retrospectivo**, não uma estimativa da precisão real da empresa. A tabela apresenta MAE por unidade; MAPE e cobertura usam pares comuns. Não mistura `un`, `kg` e `L` no MAE.

In [7]:
training_by_product = defaultdict(list)
for row in files["historico_treino.csv"]:
    if row["tipo"] == "VENDAS" and row["periodo"] <= "2026-06" and utc(row["disponivel_em"]) <= month_start("2026-07"):
        training_by_product[row["produto"]].append(row)
dataset_forecasts, dataset_pairs = {}, {}
for product, history_rows in sorted(training_by_product.items()):
    history_rows.sort(key=lambda r: r["periodo"])
    check([r["periodo"] for r in history_rows] == [plus_month("2024-07", i) for i in range(24)],
          f"24 meses consecutivos: {product}")
    values = [Decimal(r["quantidade"]) for r in history_rows]
    mm = moving_average(values[-3:], 3)
    ets = list(map(Decimal, ets_reference(values, 3)["predictions"]))
    dataset_forecasts[product] = (values, mm, ets)
    dataset_pairs[product] = [(mm[h], ets[h], actual.get((product, plus_month("2026-07", h)))) for h in range(3)]
summary_by_unit = {}
for unit in sorted({p["unidade"] for p in products.values()}):
    pairs = [pair for product, values in dataset_pairs.items() if products[product]["unidade"] == unit for pair in values]
    summary_by_unit[unit] = common_metrics(pairs)
    m = summary_by_unit[unit]
    print(f"{unit:2} | pares {m['pares']:2}/{m['candidatos']:2} | MAE MM {m['maeMM']:>8} / ETS {m['maeETS']:>8} | MAPE MM {m['mapeMM']:>8}% / ETS {m['mapeETS']:>8}%")
check(sum(len(p) for p in dataset_pairs.values()) == 72, "24 produtos × 3 horizontes = 72 candidatos")
check(sum(m["pares"] for m in summary_by_unit.values()) == 72, "Os 72 realizados de julho–setembro estão presentes")
print("Pares MAPE:", sum(m["paresMape"] for m in summary_by_unit.values()), "| Zeros excluídos apenas do MAPE:", sum(m["zeros"] for m in summary_by_unit.values()))

L  | pares  6/ 6 | MAE MM  69.6682 / ETS  85.4005 | MAPE MM  10.0501% / ETS  14.2020%
kg | pares  9/ 9 | MAE MM  39.2490 / ETS  35.6120 | MAPE MM   5.2538% / ETS   5.0153%
un | pares 57/57 | MAE MM  71.1228 / ETS  66.8894 | MAPE MM  18.7322% / ETS  20.0118%
Pares MAPE: 61 | Zeros excluídos apenas do MAPE: 11


## 8. Comparação com as classes Java reais

A verificação seguinte executa `MovingAverageEngine`, `EtsAanEngine`, `CommonMetrics` e `MonitoringMetrics` compilados em `backend/target/classes`. Não são substituídos por implementações simuladas.

Além dos dados sintéticos, comparamos casos extremos, vinte janelas pseudoaleatórias e as oito referências ETS. O programa Java temporário é apagado no final. Se o backend não estiver compilado ou estiver desatualizado, a célula falha com uma instrução de compilação.

In [8]:
java = shutil.which("java")
if java is None:
    raise RuntimeError("Instala Java 17+ e compila o backend antes desta célula.")
java_names = {
    "forecasting/MovingAverageEngine": "forecasting/MovingAverageEngine",
    "forecasting/EtsAanEngine": "forecasting/EtsAanEngine",
    "evaluation/CommonMetrics": "evaluation/CommonMetrics",
    "monitoring/MonitoringMetrics": "monitoring/MonitoringMetrics",
    "evaluation/BacktestDtos": "evaluation/BacktestDtos",
    "monitoring/MonitoringDtos": "monitoring/MonitoringDtos",
}
for name in java_names:
    source = REPO / f"backend/src/main/java/pt/tiagodias/sop/{name}.java"
    compiled = CLASSES / f"pt/tiagodias/sop/{name}.class"
    if not compiled.is_file() or compiled.stat().st_mtime_ns < source.stat().st_mtime_ns:
        raise RuntimeError("Backend por compilar/atualizar: executa cd backend && ./mvnw -q -DskipTests compile")

JAVA_SOURCE = r"""
import java.math.*;
import java.util.*;
import java.io.*;
import java.util.stream.*;
import pt.tiagodias.sop.forecasting.*;
import pt.tiagodias.sop.evaluation.*;
import pt.tiagodias.sop.monitoring.*;
public class NexoPlanNotebookProbe {
  static BigDecimal d(String value) { return value.equals("~") ? null : new BigDecimal(value); }
  static List<BigDecimal> values(String value) {
    return Arrays.stream(value.split(",")).map(BigDecimal::new).toList();
  }
  static String join(List<BigDecimal> values) {
    return values.stream().map(BigDecimal::toPlainString).collect(Collectors.joining(","));
  }
  static MonitoringDtos.Celula cell(String name, MonitoringDtos.Operacional op) {
    return new MonitoringDtos.Celula("1","NX-TEST","un","2026-07",1,"100","100",null,null,null,
      true,false,"SEM_REALIZADO",op,null);
  }
  public static void main(String[] args) throws Exception {
    try (var reader = new BufferedReader(new InputStreamReader(System.in))) {
      for (String line; (line = reader.readLine()) != null;) {
        var parts = line.split("\t", -1);
        String kind = parts[0], id = parts[1];
        if (kind.equals("MM")) {
          var input = values(parts[3]); var original = List.copyOf(input);
          var result = new MovingAverageEngine().forecast(input, Integer.parseInt(parts[2]));
          if (!input.equals(original)) throw new AssertionError("Histórico alterado");
          System.out.println(kind+"\t"+id+"\t"+join(result));
        } else if (kind.equals("ETS")) {
          var engine = new EtsAanEngine(); var input = values(parts[3]); var original = List.copyOf(input);
          var fit = parts[4].equals("~") ? engine.fit(input, Integer.parseInt(parts[2])) :
            engine.fitFixed(input, Integer.parseInt(parts[2]), Double.parseDouble(parts[4]), Double.parseDouble(parts[5]));
          if (!input.equals(original)) throw new AssertionError("Histórico alterado");
          System.out.println(kind+"\t"+id+"\t"+join(fit.predictions())+"\t"+
            fit.parameters().get("alpha")+"\t"+fit.parameters().get("beta")+"\t"+fit.parameters().get("sseTreino"));
        } else if (kind.equals("MET")) {
          var pairs = new ArrayList<CommonMetrics.Pair>();
          if (!parts[2].isEmpty()) for (var triple : parts[2].split(";")) {
            var numbers = triple.split(",");
            pairs.add(new CommonMetrics.Pair(d(numbers[0]),d(numbers[1]),d(numbers[2])));
          }
          var m = CommonMetrics.calculate(pairs);
          System.out.println(kind+"\t"+id+"\t"+m.candidatos()+"\t"+m.paresComparaveis()+"\t"+
            m.paresMape()+"\t"+m.zerosExcluidosMape()+"\t"+m.realizadosEmFalta()+"\t"+
            m.semPrevisaoMediaMovel()+"\t"+m.semPrevisaoEts()+"\t"+m.coberturaPercentagem()+"\t"+
            m.maeMediaMovel()+"\t"+m.maeEts()+"\t"+m.mapeMediaMovel()+"\t"+m.mapeEts());
        } else if (kind.equals("OP")) {
          var a = new MonitoringDtos.Operacional("1",null,"200","180",31,1,"120","100","2",null,null,null,"20");
          var b = new MonitoringDtos.Operacional("2",null,"30","30",28,2,"80","100","3",null,null,null,"0");
          var result = MonitoringMetrics.operation(List.of(cell("a",a),cell("b",b)));
          System.out.println(kind+"\t"+id+"\t"+result.nivelServicoPercentagem()+"\t"+
            result.ruturaPercentagem()+"\t"+result.excessoStockMedio()+"\t"+result.unidadesPerdidasTotal());
          var zero = new MonitoringDtos.Operacional("3",null,"0","0",0,0,null,null,null,null,null,null,null);
          var empty = MonitoringMetrics.operation(List.of(cell("c",zero)));
          System.out.println("OP\tzero\t"+empty.nivelServicoPercentagem()+"\t"+empty.ruturaPercentagem()+"\t"+
            empty.excessoStockMedio()+"\t"+empty.unidadesPerdidasTotal());
        } else throw new IllegalArgumentException(kind);
      }
    }
  }
}
"""

commands, expected_java = [], {}
def forecast_command(kind, name, values, horizon=3, fixed=None):
    serialized = ",".join(map(str, values))
    row = [kind, name, str(horizon), serialized]
    if kind == "ETS":
        row.extend(map(str, fixed) if fixed else ["~", "~"])
        expected_java[(kind, name)] = ets_reference(values, horizon, fixed)
    else:
        expected_java[(kind, name)] = list(map(q4, moving_average(values, horizon)))
    commands.append("\t".join(row))

for name, values, horizon, _ in ma_cases:
    forecast_command("MM", name, values, horizon)
rng = random.Random(SEED)
for i in range(20):
    values = [Decimal(rng.randrange(0, 100000)) / 100 for _ in range(rng.randint(1, 12))]
    forecast_command("MM", f"janela-{i}", values, rng.randint(1, 12))
for case in fixture["cases"]:
    forecast_command("ETS", case["name"], case["history"], fixed=(case["alpha"], case["beta"]) if case["fixed"] else None)
for product, (values, mm, ets) in dataset_forecasts.items():
    forecast_command("MM", product, values[-3:])
    forecast_command("ETS", product, values)
all_metric_cases = {**metric_cases, **dataset_pairs}
for name, pairs in all_metric_cases.items():
    serialized = ";".join(",".join("~" if value is None else str(value) for value in pair) for pair in pairs)
    commands.append("\t".join(["MET", name, serialized]))
    expected_java[("MET", name)] = common_metrics(pairs)
commands.append("OP\tponderada")
with tempfile.TemporaryDirectory(prefix="nexoplan-notebook-") as temporary:
    probe = Path(temporary) / "NexoPlanNotebookProbe.java"
    probe.write_text(JAVA_SOURCE, encoding="utf-8")
    run = subprocess.run([java, "--class-path", str(CLASSES), str(probe)], input="\n".join(commands)+"\n",
                         text=True, capture_output=True, timeout=90)
if run.returncode != 0:
    raise RuntimeError("Falha ao executar classes Java:\n" + run.stderr[-3000:])
java_results = {}
for line in run.stdout.splitlines():
    parts = line.split("\t")
    java_results[(parts[0], parts[1])] = parts[2:]
check(len(java_results) == len(commands)+1, "Todas as chamadas Java produziram resultados")
print("Chamadas aos métodos reais:", dict(Counter(kind for kind, _ in java_results)))

Chamadas aos métodos reais: {'MM': 48, 'ETS': 32, 'MET': 29, 'OP': 2}


In [9]:
for (kind, name), expected in expected_java.items():
    found = java_results[(kind, name)]
    if kind == "MM":
        check(found[0].split(",") == expected, f"Java/Python MM: {name}")
    elif kind == "ETS":
        check(found[0].split(",") == expected["predictions"], f"Java/Python ETS: {name}")
        check(math.isclose(float(found[1]), expected["alpha"], abs_tol=1e-12) and
              math.isclose(float(found[2]), expected["beta"], abs_tol=1e-12), f"Coeficientes ETS Java/Python: {name}")
        check(math.isclose(float(found[3]), expected["sse"], rel_tol=1e-10, abs_tol=1e-8), f"SSE Java/Python: {name}")
    else:
        keys = ["candidatos", "pares", "paresMape", "zeros", "emFalta", "semMM", "semETS",
                "cobertura", "maeMM", "maeETS", "mapeMM", "mapeETS"]
        normalized = [int(value) if i < 7 else None if value == "null" else value for i, value in enumerate(found)]
        check(dict(zip(keys, normalized)) == expected, f"Métricas Java/Python: {name}")
check(java_results[("OP", "ponderada")] == [q4(Decimal(210)/230*100), q4(Decimal(3)/59*100), "10.0000", "5.0000"],
      "Indicadores operacionais Java: rácios ponderados, excesso médio e perdas totais")
check(java_results[("OP", "zero")] == ["null"] * 4, "Indicadores sem denominador/dados: indisponíveis")
print("Comparações com Java: todas coincidem a quatro casas decimais.")
print("Serviço ponderado:", java_results[("OP", "ponderada")][0]+"%", "| Rutura ponderada:", java_results[("OP", "ponderada")][1]+"%")

Comparações com Java: todas coincidem a quatro casas decimais.
Serviço ponderado: 91.3043% | Rutura ponderada: 5.0847%


## 9. Resultado e limites

A execução só conclui se todas as verificações passarem. Uma asserção falhada interrompe a experiência; não é convertida em sucesso.

**Verificado:** integridade dos CSV versionados, referências numéricas, preservação do histórico de entrada, regras temporais ilustradas e comparação direta dos motores/métricas Java. O MAE permanece separado por unidade.

**Fora deste ensaio:** estado da base de dados local, ingestão CSV pela API, sessões/perfis, SQL e publicação/aprovação, fugas reais entre cortes numa execução REST, qualidade preditiva em dados reais e capacidade em carga. Estes pontos exigem as suites de integração e os testes de navegador.

**Execução registada:** células Python executadas sequencialmente com a biblioteca padrão. Não havia `nbclient`/kernel Jupyter no ambiente; a integração com um kernel Jupyter não foi testada. O notebook não depende de magias IPython.

In [10]:
print(f"PASSOU: {len(checks)} verificações concluídas.")
print(f"Dados: {len(products)} produtos sintéticos; 72 candidatos no corte julho–setembro.")
print(f"Java: {len(java_results)} resultados reais comparados; sem acesso à base de dados.")
print("Limite: 1 corte, dados sintéticos e classes compiladas locais; não constitui certificação de segurança.")

PASSOU: 276 verificações concluídas.
Dados: 24 produtos sintéticos; 72 candidatos no corte julho–setembro.
Java: 111 resultados reais comparados; sem acesso à base de dados.
Limite: 1 corte, dados sintéticos e classes compiladas locais; não constitui certificação de segurança.
